In [0]:
# Config
CATALOG = 'fahrlytics'
BRONZE = f'{CATALOG}.bronze'
BASE_PATH = f'/Volumes/{CATALOG}/bronze/raw_data'

CUSTOMERS_PATH = f'{BASE_PATH}/customers/'
DRIVERS_PATH   = f'{BASE_PATH}/drivers/'
PAYMENTS_PATH  = f'{BASE_PATH}/payments/'

spark.sql(f'USE CATALOG {CATALOG}')

In [0]:
# Cell 2: Target Delta Tables Definition
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE}.customers (
        customer_id             STRING,
        name                    STRING,
        city                    STRING,
        signup_date             DATE,
        device_type             STRING,
        referral_source         STRING,
        rating                  DOUBLE,
        is_churned              BOOLEAN,
        churn_date              DATE,
        _ingested_at            TIMESTAMP
    )
    USING DELTA
    COMMENT 'Raw customer snapshots ingested via COPY INTO'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE}.drivers (
        driver_id        STRING,
        name             STRING,
        city             STRING,
        vehicle_type     STRING,
        rating           DOUBLE,
        trips_completed  INT,
        joining_date     DATE,
        status           STRING,
        _ingested_at     TIMESTAMP
    )
    USING DELTA
    COMMENT 'Raw driver master ledger'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE}.payments (
        payment_id        STRING,
        trip_id           STRING,
        customer_id       STRING,
        amount            DOUBLE,
        currency          STRING,
        payment_method    STRING,
        payment_timestamp TIMESTAMP,
        payment_status    STRING,
        card_hash         STRING,
        is_fraud_card     BOOLEAN,
        _ingested_at      TIMESTAMP
    )
    USING DELTA
    COMMENT 'Raw payments ledger with currency and fraud flags'
""")

In [0]:
# COPY INTO Customers
spark.sql(f"""
    COPY INTO {BRONZE}.customers
    FROM (
        SELECT 
            customer_id,
            name,
            city,
            CAST(signup_date AS DATE)   AS signup_date,
            device_type,
            referral_source,
            CAST(rating AS DOUBLE)      AS rating,
            CAST(is_churned AS BOOLEAN) AS is_churned,
            CAST(churn_date AS DATE)    AS churn_date,
            current_timestamp()         AS _ingested_at
        FROM '{CUSTOMERS_PATH}'
    )
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'nullValue' = '')
    COPY_OPTIONS ('mergeSchema' = 'true', 'force' = 'false')
""")

In [0]:
# COPY INTO Drivers
spark.sql(f"""
    COPY INTO {BRONZE}.drivers
    FROM (
        SELECT 
            driver_id,
            name,
            city,
            vehicle_type,
            CAST(rating AS DOUBLE)       AS rating,
            CAST(trips_completed AS INT) AS trips_completed,
            CAST(joining_date AS DATE)   AS joining_date,
            status,
            current_timestamp()          AS _ingested_at
        FROM '{DRIVERS_PATH}'
    )
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'nullValue' = '')
    COPY_OPTIONS ('mergeSchema' = 'true', 'force' = 'false')
""")

In [0]:
# COPY INTO Payments
spark.sql(f"""
    COPY INTO {BRONZE}.payments
    FROM (
        SELECT 
            payment_id,
            trip_id,
            customer_id,
            CAST(amount AS DOUBLE)               AS amount,
            currency,
            payment_method,
            CAST(payment_timestamp AS TIMESTAMP) AS payment_timestamp,
            payment_status,
            card_hash,
            CAST(is_fraud_card AS BOOLEAN)       AS is_fraud_card,
            current_timestamp()                  AS _ingested_at
        FROM '{PAYMENTS_PATH}'
    )
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'nullValue' = '')
    COPY_OPTIONS ('mergeSchema' = 'true', 'force' = 'false')
""")